# Krycklan GLUE results analysis

Saved outputs may be from an earlier analysis. Rerun the cells to refresh them. The time-series section reruns every selected simulation and can be computationally expensive.

## Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()
sns.set_context('notebook', font_scale=1.5)

#%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
results_folder = os.environ.get('SPAFHY_RESULTS_FOLDER')
if not results_folder or not results_folder.strip():
    raise EnvironmentError('SPAFHY_RESULTS_FOLDER is not set. Define it in .env or the environment.')
#glue_results_csv = Path(results_folder).expanduser() / 'krycklan' / 'glue_krycklan' / '20261105131819' / 'glue_results.csv'
glue_results_csv = Path(results_folder).expanduser() / 'krycklan' / 'glue_krycklan' / '20261106133122' / 'glue_results.csv'
soil_type = 'Medium'
metrics = ['kge_tot', 'rmse_tot', 'mbe_tot', 'kge_sbsrf', 'rmse_sbsrf', 'mbe_sbsrf']
parameter_columns = [f'kmax_{soil_type}', f'kmin_{soil_type}', f'f_{soil_type}']

data = pd.read_csv(glue_results_csv)
required_columns = ['sample_id', *parameter_columns, *metrics]
missing = set(required_columns) - set(data.columns)
if missing:
    raise ValueError(f'Missing columns in {glue_results_csv}: {sorted(missing)}')
if data.empty:
    raise ValueError(f'No samples in {glue_results_csv}')
if data['sample_id'].isna().any() or data['sample_id'].duplicated().any():
    raise ValueError('sample_id values must be non-null and unique')
data[parameter_columns + metrics] = data[parameter_columns + metrics].apply(pd.to_numeric, errors='raise')
if not np.isfinite(data[parameter_columns + metrics].to_numpy()).all():
    raise ValueError('Parameter and metric values must be finite')
if (data[[f'kmax_{soil_type}', f'kmin_{soil_type}']] <= 0).any().any():
    raise ValueError('kmax and kmin must be positive')
data = data.set_index('sample_id', drop=False)
print(f'Loaded {len(data)} samples from {glue_results_csv}')
print(f'Recorded kmin values: {data[f"kmin_{soil_type}"].unique()}')

# Calculate rank for each sample_id and metric

In [41]:
for metric_name in metrics:
    values = data[metric_name].abs() if metric_name.startswith('mbe_') else data[metric_name]
    ascending = not metric_name.startswith('kge_')
    data[f'rank_{metric_name}'] = values.rank(method='min', ascending=ascending)

In [42]:
# Adjust these weights to change the importance of each metric in the final ranking.
w_kge = 0.5
w_rmse = 0.25
w_mbe = 0.25
N = len(data)
for suffix in ('tot', 'sbsrf'):
    data[f'points_{suffix}'] = (
        w_kge * (N - data[f'rank_kge_{suffix}'])
        + w_rmse * (N - data[f'rank_rmse_{suffix}'])
        + w_mbe * (N - data[f'rank_mbe_{suffix}'])
    )
    data[f'rank_{suffix}'] = data[f'points_{suffix}'].rank(method='min', ascending=False)

# GLUE analysis

Choose the number of simulations to retain for total and subsurface runoff
separately. Each set is ranked by KGE (highest first); the KGE of the last
retained simulation is shown as the threshold. Ties at the boundary are
resolved by CSV order. `color_metric` controls the parameter-plot colors.


In [ ]:
color_metric = 'kge'  # kge, rmse, mbe, or points; used for plot colors
n_sims_tot = 100
n_sims_sbsrf = 100
if color_metric not in ('kge', 'rmse', 'mbe', 'points'):
    raise ValueError(f'Unsupported color metric: {color_metric}')

for suffix, count in (('tot', n_sims_tot), ('sbsrf', n_sims_sbsrf)):
    if not isinstance(count, int) or count < 1:
        raise ValueError(f'n_sims_{suffix} must be a positive integer')
    if count > len(data):
        raise ValueError(f'n_sims_{suffix}={count} exceeds the {len(data)} available samples')

kge_cutoff_tot = data['kge_tot'].nlargest(n_sims_tot, keep='first').iloc[-1]
kge_cutoff_sbsrf = data['kge_sbsrf'].nlargest(n_sims_sbsrf, keep='first').iloc[-1]


In [ ]:
def metric_values(frame, metric_name, suffix):
    values = frame[f'{metric_name}_{suffix}']
    return values.abs() if metric_name == 'mbe' else values


fig, ax = plt.subplots(1, 2, figsize=(14, 6))
fig.subplots_adjust(wspace=0.3)
for idx, suffix in enumerate(['tot', 'sbsrf']):
    values = data[f'kge_{suffix}']
    cutoff = kge_cutoff_tot if suffix == 'tot' else kge_cutoff_sbsrf
    num_sims = n_sims_tot if suffix == 'tot' else n_sims_sbsrf
    sns.histplot(x=values, bins=100, kde=True, ax=ax[idx])
    ax[idx].axvline(cutoff, color='red', linestyle='--', label=f'KGE threshold: {cutoff:.3f}')
    ax[idx].set_xlabel(f'kge_{suffix}')
    ax[idx].set_title(f'{suffix.upper()} KGE ({num_sims} simulations selected)')
    ax[idx].legend()

In [ ]:
suitable_simulations_tot = data.nlargest(n_sims_tot, 'kge_tot', keep='first')
suitable_simulations_sbsrf = data.nlargest(n_sims_sbsrf, 'kge_sbsrf', keep='first')

print(f'Total runoff: selected {len(suitable_simulations_tot)} simulations; KGE threshold={kge_cutoff_tot:.3f}')
print(f'Subsurface runoff: selected {len(suitable_simulations_sbsrf)} simulations; KGE threshold={kge_cutoff_sbsrf:.3f}')

## Visualize sampled kmax and f

The top row shows simulations selected by the GLUE cutoff. The bottom row
shows all simulations over the full current GLUE search ranges: kmax from
1e-4 to 1e-2 m/s and f from 5 to 20. The red rectangle marks the kmax/f
extent of the suitable simulations. The current GLUE runner fixes `kmin`;
historical CSVs may contain varying `kmin` values.


In [27]:
print(f'kmax range in CSV: {data[f"kmax_{soil_type}"].min():.3g}–{data[f"kmax_{soil_type}"].max():.3g} m/s')
print(f'f range in CSV: {data[f"f_{soil_type}"].min():.3g}–{data[f"f_{soil_type}"].max():.3g}')

In [ ]:
F_BOUNDS = (5.0, 20.0)
KMAX_BOUNDS = (1e-4, 0.01)

fig, axes = plt.subplots(2, 2, figsize=(14, 12), sharex='row', sharey='row')
fig.subplots_adjust(wspace=0.4, hspace=0.3)

for row, row_axes in enumerate(axes):
    for ax, suitable, suffix in zip(
        row_axes, [suitable_simulations_tot, suitable_simulations_sbsrf], ['tot', 'sbsrf']
    ):
        full_range = row == 1
        plot_data = data if full_range else suitable
        colors = metric_values(plot_data, color_metric, suffix)
        scatter = ax.scatter(
            plot_data[f'kmax_{soil_type}'], plot_data[f'f_{soil_type}'],
            c=colors, cmap='viridis', s=20 if full_range else 50,
            alpha=0.25 if full_range else 0.6
        )
        ax.set_xscale('log')
        if full_range:
            f_padding = (F_BOUNDS[1] - F_BOUNDS[0]) * 0.05
            ax.set_xlim(KMAX_BOUNDS[0] / 10**0.05, KMAX_BOUNDS[1] * 10**0.05)
            ax.set_ylim(F_BOUNDS[0] - f_padding, F_BOUNDS[1] + f_padding)
            if not suitable.empty:
                x_min, x_max = suitable[f'kmax_{soil_type}'].min(), suitable[f'kmax_{soil_type}'].max()
                y_min, y_max = suitable[f'f_{soil_type}'].min(), suitable[f'f_{soil_type}'].max()
                ax.plot(
                    [x_min, x_max, x_max, x_min, x_min],
                    [y_min, y_min, y_max, y_max, y_min],
                    color='red', linestyle='--', linewidth=1.5, label='Suitable extent'
                )
                ax.legend(loc='upper right')
            else:
                ax.text(0.5, 0.5, 'No suitable simulations', transform=ax.transAxes, ha='center')
            ax.set_title(f'{suffix.upper()} all simulations with suitable extent')
        else:
            ax.set_title(f'{suffix.upper()} suitable simulations (n={len(suitable)})')

        ax.set_xlabel('kmax (m/s)')
        ax.set_ylabel('f')
        color_label = f'abs(mbe_{suffix})' if color_metric == 'mbe' else f'{color_metric}_{suffix}'
        fig.colorbar(scatter, ax=ax, label=color_label)

## Visualize Ksat curves of suitable simulations

Using the current soil-generator defaults, conductivity is `kmax` down to
the constant surface depth, then decays as

$$K_{sat} = (K_{max} - K_{min})e^{-f(d-d_c)} + K_{min}, \quad d > d_c.$$

Here $d$ is positive depth and $d_c$ is the constant surface depth. Historical
results may have used different soil-generator settings.

In [ ]:
from scripts.create_soil_params import _CONST_SURF_KMAX, _MAX_DEPTH

constant_depth = _CONST_SURF_KMAX[soil_type]
depth = np.linspace(0, _MAX_DEPTH[soil_type], 100)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, suitable, title in zip(
    axes,
    [suitable_simulations_tot, suitable_simulations_sbsrf],
    ['TOT suitable simulations', 'SBSRF suitable simulations']
):
    for _, row in suitable.iterrows():
        kmin = row[f'kmin_{soil_type}']
        kmax = row[f'kmax_{soil_type}']
        f = row[f'f_{soil_type}']
        below_const = np.maximum(depth - constant_depth, 0)
        k_sat = (kmax - kmin) * np.exp(-f * below_const) + kmin
        ax.plot(np.log10(k_sat), -depth, color='grey', alpha=0.1)
    ax.set_xlabel('log10(k$_{sat}$ [m/s])')
    ax.set_ylabel('z (m)')
    ax.set_title(title)

# Show best simulation result time series

In [30]:
from run_best_krycklan_parameters import run_batch_simulations, run_single_simulation

project_folder = os.environ.get('PROJECT_FOLDER')
if not project_folder or not project_folder.strip():
    raise EnvironmentError('PROJECT_FOLDER is not set. Define it in .env or the environment.')

model_root = Path(project_folder).expanduser() / 'krycklan'
print(f'Model input root: {model_root}')

Model root: /Users/opa/SpaFHy/krycklan/SpaFHy_RUNS/krycklan
Output folder for netcdf will be: <glue_results_folder>/simulations


## Load observations

In [31]:
from read_runoff import read_runoff

catchment_no = 2  # C2 catchment
runoff_folder = str(model_root / 'obs' / 'DISCHARGE')
q_obs = read_runoff(catchment_no, runoff_folder)

In [ ]:
catchment = 'krycklan'
if suitable_simulations_tot.empty or suitable_simulations_sbsrf.empty:
    raise ValueError('No suitable simulations for one or both runoff components; adjust the cutoffs.')

print(f'Running simulations for {len(suitable_simulations_tot)} suitable total runoff models...')
results_tot = run_batch_simulations(
    suitable_simulations_tot,
    folder=str(model_root),
    q_obs=q_obs['Q'],
    output_folder=None,
    save_ncf=False,
    catchment=catchment,
    catchment_no=catchment_no,
    soil_type=soil_type,
    skip_existing=True,
    verbose=False,
)

print(f'Running simulations for {len(suitable_simulations_sbsrf)} suitable subsurface runoff models...')
results_sbsrf = run_batch_simulations(
    suitable_simulations_sbsrf,
    folder=str(model_root),
    q_obs=q_obs['Q'],
    output_folder=None,
    save_ncf=False,
    catchment=catchment,
    catchment_no=catchment_no,
    soil_type=soil_type,
    skip_existing=True,
    verbose=False,
)

print(f'Total simulations completed: {len(results_tot)}')
print(f'Subsurface simulations completed: {len(results_sbsrf)}')

In [ ]:
def extract_best_and_ranges(results_dict, suffix):
    """Extract the minimum-RMSE simulation and metric ranges."""
    q_dfs = {}
    params_list = []
    idx_list = []

    for idx, result in results_dict.items():
        if result is not None:
            q_df, params, _ = result
            q_dfs[idx] = q_df
            params_list.append(params)
            idx_list.append(idx)

    if not params_list:
        raise ValueError(f'No completed {suffix} simulations to analyze')
    params_df_all = pd.DataFrame(params_list, index=idx_list)
    best_idx = params_df_all[f'rmse_{suffix}'].idxmin()
    best_q_df = q_dfs[best_idx]
    best_params = params_df_all.loc[best_idx].to_dict()
    ranges = {name: (params_df_all[name].min(), params_df_all[name].max()) for name in metrics}
    return best_idx, best_q_df, best_params, ranges, q_dfs


best_idx_tot, best_q_df_tot, best_params_tot, ranges_tot, q_dfs_tot = extract_best_and_ranges(
    results_tot, 'tot'
)
best_idx_sbsrf, best_q_df_sbsrf, best_params_sbsrf, ranges_sbsrf, q_dfs_sbsrf = extract_best_and_ranges(
    results_sbsrf, 'sbsrf'
)
print(f"Best total runoff simulation: KGE={best_params_tot['kge_tot']:.2f}, RMSE={best_params_tot['rmse_tot']:.2f}")
print(f"Best subsurface runoff simulation: KGE={best_params_sbsrf['kge_sbsrf']:.2f}, RMSE={best_params_sbsrf['rmse_sbsrf']:.2f}")

In [ ]:
best_params_tot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('SpaFHy Model - Best Suitable Simulations vs Observed Runoff', fontsize=14, fontweight='bold')

runoff_types = [
    ('tot', 'q_sim', best_idx_tot, best_params_tot, ranges_tot, q_dfs_tot, best_q_df_tot, axes[0]),
    ('sbsrf', 'q_sim_sbsrf', best_idx_sbsrf, best_params_sbsrf, ranges_sbsrf, q_dfs_sbsrf, best_q_df_sbsrf, axes[1])
]

for runoff_type, q_col, best_idx, best_params, ranges, q_dfs, best_q_df, ax in runoff_types:
    overland_flow = best_q_df['q_sim'] - best_q_df['q_sim_sbsrf']
    ax.stackplot(
        best_q_df.index,
        best_q_df['q_sim_sbsrf'],
        overland_flow,
        labels=['Subsurface', 'Overland flow'],
        colors=['tab:orange', 'tab:green'],
        alpha=0.8,
    )

    for idx, q_df in q_dfs.items():
        if idx != best_idx:
            ax.plot(q_df.index, q_df[q_col], color='lightgray', linewidth=1, alpha=0.75, label='_nolegend_')

    ax.plot(best_q_df.index, best_q_df['q_obs'], color='black', linewidth=2, label='Observed', zorder=4)

    kge_val = best_params[f'kge_{runoff_type}']
    kge_min, kge_max = ranges[f'kge_{runoff_type}']
    rmse_val = best_params[f'rmse_{runoff_type}']
    rmse_min, rmse_max = ranges[f'rmse_{runoff_type}']
    mbe_val = best_params[f'mbe_{runoff_type}']
    mbe_min, mbe_max = ranges[f'mbe_{runoff_type}']

    title = (f"{runoff_type.capitalize()} Runoff — "
             f"KGE: {kge_val:.2f} ({kge_min:.2f}–{kge_max:.2f}) | "
             f"RMSE: {rmse_val:.1f} ({rmse_min:.1f}–{rmse_max:.1f}) | "
             f"MBE: {mbe_val:.1f} ({mbe_min:.1f}–{mbe_max:.1f})")
    ax.set_title(title, fontsize=11, loc='left')

    ax.set_ylabel('Runoff [mm/d]')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', framealpha=0.9)

axes[1].set_xlabel('Time')
plt.tight_layout()
plt.show()

print("\nVisualization complete! All suitable simulations shown as light gray swarm.")
print("Stacked best-simulation components and observed runoff are highlighted.")